In [2]:
import pandas as pd
import numpy as np


def add_block_id(df):
    """Δημιουργεί block_id: νέο block κάθε φορά που αλλάζει το Class
    ή το PathOrder (ώστε να μην ενωθούν trips διαφορετικών PathOrder)."""
    df = df.copy()
    class_change = (df['Class'] != df['Class'].shift(1)).astype(int)
    path_change = (df['PathOrder'] != df['PathOrder'].shift(1)).astype(int)
    df['block_id'] = (class_change | path_change).cumsum()
    return df


def split_blocks(df, train_frac=0.7, val_frac=0.1, meta_frac=0.1):
    """
    Μέσα σε κάθε block, κόβει διαδοχικά 4 κομμάτια:
    - convtran_train  (εκπαίδευση κάθε ConvTran)
    - convtran_val    (early stopping -- ΝΕΟ)
    - meta_train      (εκπαίδευση του τελικού classifier πάνω σε embeddings)
    - test            (τελικό, κοινό test set)
    Η σειρά των γραμμών ΔΕΝ αλλάζει μέσα σε κάθε κομμάτι.
    """
    df = df.copy()
    df['split'] = None

    for block, group in df.groupby('block_id'):
        n = len(group)
        n_train = int(n * train_frac)
        n_val = int(n * val_frac)
        n_meta = int(n * meta_frac)

        idx = group.index
        df.loc[idx[:n_train], 'split'] = 'convtran_train'
        df.loc[idx[n_train:n_train + n_val], 'split'] = 'convtran_val'
        df.loc[idx[n_train + n_val:n_train + n_val + n_meta], 'split'] = 'meta_train'
        df.loc[idx[n_train + n_val + n_meta:], 'split'] = 'test'

    return df


# ---------- Εφαρμογή σε κάθε ομάδα ----------
for name in ['cleaned_1', 'cleaned_2', 'cleaned_3']:  # ABC, DEF, GHIJ
    df = pd.read_csv(f'{name}.csv')
    df = add_block_id(df)
    df = split_blocks(df, train_frac=0.7, val_frac=0.1, meta_frac=0.1)
    print(f'=== {name} ===')
    print(df.groupby(['Class', 'split']).size().unstack(fill_value=0))
    print()

    df.to_csv(f'{name}_split.csv', index=False)

=== cleaned_1 ===
split  convtran_train  convtran_val  meta_train  test
Class                                                
0                5067           723         723   727
1                9003          1284        1284  1293
2                5249           749         749   753

=== cleaned_2 ===
split  convtran_train  convtran_val  meta_train  test
Class                                                
0                9270          1324        1324  1326
1                5904           843         843   846
2                7707          1101        1101  1103

=== cleaned_3 ===
split  convtran_train  convtran_val  meta_train  test
Class                                                
0                5244           748         748   752
1                6915           987         987   991
2                5464           779         779   786
3                6231           889         889   895

